In [1]:
# Ensure project root on sys.path and load .env
import os
import sys
from pathlib import Path

# Explicit project root
project_root = Path("/home/snt/projects_lujun/agi_index_tournament")

if project_root and (project_root / "src" / "agi_toolkit").exists():
    if str(project_root / "src") not in sys.path:
        sys.path.insert(0, str(project_root / "src"))
else:
    # Fallback detection if path changes
    root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for base in root_candidates:
        if (base / "src" / "agi_toolkit").exists():
            project_root = base
            if str(project_root / "src") not in sys.path:
                sys.path.insert(0, str(project_root / "src"))
            break

# Load .env if present at project root
try:
    from dotenv import load_dotenv  # optional
    dotenv_path = project_root / ".env" if project_root else None
    if dotenv_path and dotenv_path.exists():
        load_dotenv(dotenv_path=dotenv_path)
    else:
        load_dotenv()  # fallback to default search
except Exception:
    pass

OPENAI_KEY = os.getenv("OPENAI_API_KEY")

print("project_root detected:", project_root)
print("OPENAI_KEY loaded:", bool(OPENAI_KEY))

project_root detected: /home/snt/projects_lujun/agi_index_tournament
OPENAI_KEY loaded: True


## GPT Client Testing


In [2]:
# Setup: import rewriter and create a stub chatgpt_fn
from agi_toolkit.backTranRewriter import BackTranslationRewriter, BackTranslationConfig
from agi_toolkit.llm_clients import LLMConfig, create_client

def stub_chatgpt_fn(prompt: str) -> str:
    # Simple echo-style translator that marks the pivot language
    if "back to English" in prompt:
        return prompt.split('\n')[-1].strip() + " [en]"
    # extract pivot name from prompt when possible
    pivot = "pivot"
    if 'to the target language' in prompt:
        pivot = prompt.split('target language')[-1].strip().split('\n')[0].strip() or pivot
    return prompt.split('\n')[-1].strip() + f" [{pivot}]"

# Example: build a real ChatGPT client (won't be called by default)
openai_cfg = LLMConfig(
    provider="openai",
    model="gpt-4o-mini",
    api_key=OPENAI_KEY,  # replace with real key or env var
)
chatgpt_client = create_client(openai_cfg) if OPENAI_KEY else None

# Choose which chat function to use
use_real = bool(OPENAI_KEY)
chat_fn = chatgpt_client.chat if use_real else stub_chatgpt_fn
print("Using real ChatGPT API:", use_real)

config = BackTranslationConfig(
    backend='chatgpt',
    chatgpt_fn=chat_fn,  # swap to real API when OPENAI_KEY is set
    pivot_languages=['French', 'German'],
    src_lang='English',
    seed=42,
)
rewriter = BackTranslationRewriter(config)
print('Configured pivot languages:', rewriter.get_pivot_languages())

/home/snt/projects_lujun/agi_index_tournament/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using real ChatGPT API: True
Configured pivot languages: ['French', 'German']


In [3]:
# Single-round rewrite with random pivot
text = "Back translation can generate diverse paraphrases.\nIt should preserve line breaks."
rewritten, meta = rewriter.rewrite(text, rounds=1, pivot_strategy='random')
print('Original text:')
print(text)
print('Rewritten text:')
print(rewritten)
print('Per-line metadata:')
meta

Original text:
Back translation can generate diverse paraphrases.
It should preserve line breaks.
Rewritten text:
Back-translation can generate various paraphrases.  
It should preserve line breaks.
Per-line metadata:


[{'pivot': 'French',
  'forward': 'La rétrotraduction peut générer des paraphrases diverses.  \nElle devrait préserver les sauts de ligne.',
  'back': 'Back-translation can generate various paraphrases.  \nIt should preserve line breaks.'}]

## NLLB Model Testing

In [2]:
# Login to Hugging Face (requires HF token in env)
from huggingface_hub import login

hf_token = os.getenv("HF_AUTH_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face login succeeded (token provided).")
else:
    print("No HF token found in env; set HF_AUTH_TOKEN or HUGGINGFACEHUB_API_TOKEN.")

/home/snt/projects_lujun/agi_index_tournament/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face login succeeded (token provided).


In [4]:

from agi_toolkit.backTranRewriter import BackTranslationRewriter, BackTranslationConfig

nllb_cfg = BackTranslationConfig(
    backend="nllb",
    model_name="facebook/nllb-200-distilled-600M",
    pivot_languages=["fra_Latn"],  # use French and Spanish fra_Latn", "spa_Latn"
    src_lang="eng_Latn",
    hf_auth_token=hf_token,  # supply token if required
    max_length=512,
    seed=0,
)
nllb_rewriter = BackTranslationRewriter(nllb_cfg)

sample_text = "Back translation with NLLB keeps line structure. Here is another line to test."
rewritten_nllb, meta_nllb = nllb_rewriter.rewrite(
    sample_text,
    rounds=3,
    pivot_strategy="random",
)

print("HF token provided:", bool(hf_token))
print("Original:\n", sample_text)
print("\nRewritten (NLLB):\n", rewritten_nllb)
print("\nPer-line metadata:")
for row in meta_nllb:
    print(row)


HF token provided: True
Original:
 Back translation with NLLB keeps line structure. Here is another line to test.

Rewritten (NLLB):
 The translation en arrière avec NLLB maintains the structure of the line.

Per-line metadata:
{'pivot': 'fra_Latn', 'forward': 'La traduction en arrière avec NLLB maintient la structure de la ligne.', 'back': 'The translation en arrière avec NLLB maintains the structure of the line.'}
{'pivot': 'fra_Latn', 'forward': 'La traduction en arrière avec NLLB maintient la structure de la ligne.', 'back': 'The translation en arrière avec NLLB maintains the structure of the line.'}
{'pivot': 'fra_Latn', 'forward': 'La traduction en arrière avec NLLB maintient la structure de la ligne.', 'back': 'The translation en arrière avec NLLB maintains the structure of the line.'}
